#### **Installing dependencies**

In [1]:
!python3.11 -m pip install jupyterlab ipykernel
!python3.11 -m pip install pandas scikit-learn pathlib tqdm torch transformers pytrends
!python3.11 -m pip install numpy==1.26.4
# pip install spacy
!python3.11 -m pip install spacy==3.6.0
# python -m spacy download en_core_web_lg
!python3.11 -m pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_lg-3.6.0/en_core_web_lg-3.6.0-py3-none-any.whl


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 28.4 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


#### **Imports**

In [2]:
import pandas as pd
from pathlib import Path
import spacy
import numpy as np
import re
from tqdm import tqdm
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity

#### **1. Initial setup**

In [5]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR/"data"

artists_path = DATA_DIR / "artists.csv"
brands_path = DATA_DIR / "brands.csv"
songs_path = DATA_DIR / "songs_final.csv"
lyrics_path = DATA_DIR / "lyrics_final.csv"

In [6]:
artists_df = pd.read_csv(artists_path)
brands_df = pd.read_csv(brands_path)
songs_df = pd.read_csv(songs_path)
lyrics_df = pd.read_csv(lyrics_path)
mentions_df = None

**Initially** we only used **regex** for brand extraction while generating *mentions.csv*. But it has significant **limitations**, which can be **improved** with deeper NLP techniques:

| Issue | NLP solution |
| :--- | :--- |
| Simple Word Boundary (\b) | The regex \b(alias)\b is good for simple words but fails if the brand is part of a compound or hyphenated word (e.g., "Air-Jordan") or if it's an acronym that is often capitalized (e.g., "BMW"). |
| Lack of Contextual Disambiguation (False Positives) | The biggest problem. Many brand names are common English words (e.g., Apple, Amazon, Shell, Tide). The regex treats all of them as a brand mention, leading to massive false positives. |
| Your Idea: Handling Repeated/Same Context Mentions | The current code generates a new row for every single match, regardless of context. This overcounts mentions for the same semantic usage. |
| Hard-Coded Context Window (10 words) | A fixed 10-word window is arbitrary. Sometimes the relevant context (like an adjective or verb) is 1 word away; sometimes it's an entire sentence. |

#### **2. Setup and Initailization for spaCy model**

In [7]:
# Load the large English NLP model (en_core_web_lg is preferred for better word vectors)
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    print("Downloading spaCy model 'en_core_web_lg'. This may take a moment.")
    spacy.cli.download("en_core_web_lg")
    nlp = spacy.load("en_core_web_lg")

# Setup for Brand Alias Mapping
alias_to_brand = {}
aliases_list = []
for _, br in brands_df.iterrows():
    bid = br['brand_id']
    main = str(br['brand_name']).strip().lower()
    alias_list = []
    if 'brand_alias' in br and not pd.isna(br['brand_alias']):
        alias_list.extend([a.strip().lower() for a in str(br['brand_alias']).split(',') if a.strip()])
    alias_list.append(main)
    for a in alias_list:
        alias_to_brand[a] = bid
        aliases_list.append(re.escape(a))

# Longest-first sorting and master regex pattern for initial candidate extraction
aliases_list = sorted(set(aliases_list), key=lambda x: -len(x))
pattern = r'\b(' + '|'.join(aliases_list) + r')\b'
master_re = re.compile(pattern, flags=re.IGNORECASE)

print("Setup complete. spaCy model loaded and brand regex compiled.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. spaCy model loaded and brand regex compiled.


#### **3. Brand mention extraction with NER disambiguation**  
This combined step iterates through the lyrics, uses spaCy for advanced tokenization and sentence boundaries, extracts candidates using regex, and then filters them using NER/POS tags

In [8]:
# Function to perform NER/POS check for disambiguation
def is_brand_mention(token, alias_text):
    """
    Checks if a token or surrounding context supports the idea that the
    alias is being used as a proper noun/entity (a brand).
    """
    # 1. Simple check: Is it tagged as a proper noun?
    if token.pos_ in ('PROPN', 'NOUN'):
        return True

    # 2. Check for surrounding proper nouns or organizations (Contextual Clue)
    # E.g., "The latest Nike shoe" - 'Nike' is often followed by another noun/proper noun
    # Check 2 words before and after
    for i in range(max(0, token.i - 2), min(len(token.doc), token.i + 3)):
        if token.doc[i].ent_type_ in ('ORG', 'PRODUCT', 'PERSON'):
            return True

    # 3. Check for specific common-word brands (e.g., 'apple', 'shell')
    # If the alias is a common word, require a strict NER tag.
    common_word_brands = {'apple', 'shell', 'tide', 'amazon', 'guess'} # Add more as needed
    if alias_text.lower() in common_word_brands:
        # Require it to be identified as an Organization (ORG) or Product
        if token.ent_type_ in ('ORG', 'PRODUCT'):
            return True
        return False # Reject common words unless strictly classified

    # Default acceptance for less ambiguous names
    return True


mentions = []
mid = 1
tqdm.pandas(desc="Processing lyrics for brand mentions") # Enable progress bar for apply

# We use spaCy's nlp.pipe for efficient processing of large text columns
for row in tqdm(lyrics_df.itertuples(), total=len(lyrics_df)):
    text = str(row.lyrics_cleaned)
    song_id = row.song_id

    # 2.A: Advanced Tokenization & Sentence Boundary Detection (SBD)
    doc = nlp(text)

    # Use the master regex to find all candidate matches
    for m in master_re.finditer(text):
        alias = m.group(1).lower()
        start, end = m.span()

        # Find the specific token that matches the start of the alias
        matching_token = next((token for token in doc if token.idx == start), None)

        if matching_token:
            # 2.B: Disambiguation and Context (Sentence) Extraction
            if is_brand_mention(matching_token, alias):
                # Context is the entire sentence
                sentence = matching_token.sent.text.strip()
                brand_id = alias_to_brand.get(alias)

                mentions.append({
                    "mention_id": mid,
                    "song_id": song_id,
                    "brand_id": brand_id,
                    "brand_alias": alias,
                    "context": sentence, # Context is now the full sentence
                    "char_start": start,
                    "char_end": end,
                    "sentiment": None,
                    "detection_method": "nlp_regex_ner",
                    # Store sentence embedding for later clustering (Step 2.C)
                    "context_vector": matching_token.sent.vector 
                })
                mid += 1

# Create the initial DataFrame
mentions_df_raw = pd.DataFrame(mentions)
print(f"Initial raw mentions found: {len(mentions_df_raw)}")

100%|██████████| 19501/19501 [09:44<00:00, 33.37it/s]

Initial raw mentions found: 7317


#### **4. Context Clustering for Repeated Mentions**
This step implements your idea: finding and grouping mentions within the same song/brand that share the same context. We use the DBSCAN clustering algorithm on the sentence vectors stored in the previous step.

We used **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** for clustering below because:

| Feature | Description |
| :-- | :--|
| Cluster Shape | Can discover clusters of arbitrary (non-spherical) shapes. |
| Input | Requires two parameters: $\text{Eps}$ (maximum distance between samples for one to be considered as in the neighborhood of the other) and $\text{MinPts}$ (the number of samples in a neighborhood for a point to be considered as a core point). |
| Noise Handling | Explicitly identifies and labels outliers as noise (unclustered points). |
| Density | Excels at finding clusters of varying densities in the same dataset. |
| Scalability | Can be slower than K-Means, especially in high-dimensional space, due to the need for proximity queries. |

In [9]:
from sklearn.preprocessing import normalize

final_mentions = []
final_mid = 1
grouped = mentions_df_raw.groupby(['song_id', 'brand_id'])

# DBSCAN parameters
EPSILON = 0.15 
MIN_SAMPLES = 1

for (song_id, brand_id), group in tqdm(grouped, desc="Clustering contexts"):
    
    N = len(group)
    
    if N == 1:
        labels = np.array([0]) 
    else:
        # 1. Extract and Normalize Vectors
        X_raw = np.stack(group['context_vector'].values)
        X = normalize(X_raw, axis=1) # Ensure vectors have unit length
        
        # 2. Compute Distance Matrix
        # Cosine distance = 1 - Cosine similarity
        dist_matrix = 1 - cosine_similarity(X)
        
        # 3. 💡 DEFINITIVE FIX: CLAMPING FOR NUMERICAL STABILITY
        # Ensure all values are non-negative, correcting for tiny floating point errors.
        # This prevents the ValueError when metric='precomputed'.
        dist_matrix = np.maximum(dist_matrix, 0)
        
        # 4. Apply DBSCAN clustering
        db = DBSCAN(eps=EPSILON, min_samples=MIN_SAMPLES, metric='precomputed').fit(dist_matrix)
        labels = db.labels_
    
    # 5. Aggregation (Rest of the code remains the same)
    group = group.copy()
    group['cluster_label'] = labels
    
    for label in set(labels):
        cluster = group[group['cluster_label'] == label]
        
        representative_mention = cluster.iloc[0].to_dict()
        
        representative_mention['mention_id'] = final_mid
        representative_mention['mention_count'] = len(cluster)
        
        del representative_mention['context_vector']
        del representative_mention['cluster_label']
        
        final_mentions.append(representative_mention)
        final_mid += 1

mentions_df = pd.DataFrame(final_mentions)
mentions_path = DATA_DIR / "mentions.csv"
mentions_df.to_csv(mentions_path, index=False)

print(f"Final mentions found after clustering: {len(mentions_df)}")
print(f"Saved mentions.csv to: {mentions_path}")

Clustering contexts: 100%|██████████| 4761/4761 [00:03<00:00, 1537.59it/s]


Final mentions found after clustering: 5415
Saved mentions.csv to: ../data/mentions.csv


#### **5. Optimized Setiment analysis**

In [10]:
import torch
import pandas as pd
from transformers import pipeline
from tqdm.auto import tqdm

In [11]:
print("Loading sentiment pipeline (may take time)...")

# Use device=0 for GPU, device=-1 for CPU (as in your original code)
DEVICE = 0 if torch.cuda.is_available() else -1
# Load a common model optimized for social media/general text sentiment
# distilbert-base-uncased-finetuned-sst-2-english is very common, but others might be better.
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english", 
    device=DEVICE,
    batch_size=32 # Set a batch size for parallel processing (tune this)
)

# 1. Prepare Text List
# Filter out empty/NaN contexts and keep track of which rows they correspond to
valid_contexts = mentions_df['context'].astype(str).str.strip().replace('', pd.NA).dropna()
valid_indices = valid_contexts.index
    
if valid_contexts.empty:
    print("No valid contexts found for sentiment analysis.")
    mentions_df['sentiment'] = 'neutral' # Assign neutral if no text
else:
    # 2. Batch Processing
    # Pass the list of contexts to the pipeline. This is much faster.
    # The pipeline handles tokenization, truncation, and batching internally.
    print(f"Running sentiment analysis on {len(valid_contexts)} unique contexts in batches...")
        
    # Use a list of strings, not a pandas Series, for direct pipeline input
    results = sentiment_pipe(list(valid_contexts), truncation=True) 
        
    # 3. Process Results
    sentiment_map = {}
    for idx, result in zip(valid_indices, results):
        label = result['label'].lower()
        if 'positive' in label:
            sentiment_map[idx] = 'positive'
        elif 'negative' in label:
            sentiment_map[idx] = 'negative'
        else:
            # Catch any unexpected labels, default to neutral
            sentiment_map[idx] = 'neutral' 

    # 4. Update DataFrame
    # Initialize column to 'neutral' for all rows
    mentions_df['sentiment'] = 'neutral'
    # Update only the rows that had valid text
    mentions_df.loc[valid_indices, 'sentiment'] = pd.Series(sentiment_map)
    
# Save the updated DataFrame
mentions_df.to_csv(mentions_path, index=False)
print("Added sentiment column using batch processing and saved mentions.csv")

Loading sentiment pipeline (may take time)...


Device set to use cpu


Running sentiment analysis on 5415 unique contexts in batches...
Added sentiment column using batch processing and saved mentions.csv


#### **6. Build master trend downloader**

In [12]:
# Downloads Google Trends per brand and stores trend.csv under /content/trends_data/<brand>/
from pytrends.request import TrendReq
import os
pytrends = TrendReq(hl='en-US', tz=360)

BRAND_TRENDS_DIR = BASE_DIR / "trends_data"
os.makedirs(BRAND_TRENDS_DIR, exist_ok=True)

brands_list = brands_df['brand_name'].dropna().unique().tolist()
def fetch_brand_trend(brand, timeframe="2004-01-01 2025-12-31", retries=5):
    safe = brand.replace("/", "_").replace(" ", "_")
    folder = BRAND_TRENDS_DIR / safe
    os.makedirs(folder, exist_ok=True)
    out_file = folder / "trend.csv"
    if out_file.exists():
        return out_file
    for attempt in range(retries):
        try:
            pytrends.build_payload([brand], timeframe=timeframe)
            df = pytrends.interest_over_time()
            if df.empty:
                return None
            df = df.reset_index()
            # remove isPartial column if present
            df = df.drop(columns=[c for c in df.columns if "isPartial" in c], errors='ignore')
            col = [c for c in df.columns if c not in ['date']][0]
            # New Line (using Exponential Weighted Mean for better time-series smoothing):
            # Adjust 'span' (or 'halflife') to control the degree of smoothing. span=4 approximates a 4-period moving average.
            df[col + "_smooth"] = df[col].ewm(span=4, adjust=False).mean()
            df.to_csv(out_file, index=False)
            return out_file
        except Exception as e:
            print(f"Retry {attempt+1}/{retries} for {brand} due to: {e}")
            time.sleep(2 + random.random()*3)
    return None

# Run sequentially (or parallelize if needed)
downloaded = []
for b in tqdm(brands_list):
    out = fetch_brand_trend(b)
    if out:
        downloaded.append(out)
print("Downloaded trends for", len(downloaded), "brands")


  0%|          | 0/62 [00:00<?, ?it/s]/usr/local/lib/python3.11/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)
100%|██████████| 62/62 [00:00<00:00, 84.78it/s]

Downloaded trends for 62 brands


#### **7. Build brand_trends_master.csv**

In [18]:
# Merge all brand trend files into normalized master: columns (date, brand_id, brand_name, value)
rows = []
for _, r in brands_df.iterrows():
    brand = r['brand_name']
    safe = brand.replace("/", "_").replace(" ", "_")
    path = BRAND_TRENDS_DIR / safe / "trend.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path, parse_dates=['date'])
    col = [c for c in df.columns if c not in ['date']][0]
    if col.endswith('_smooth'):
        smooth_col = col
    else:
        smooth_col = col + '_smooth' if (col + '_smooth') in df.columns else col
    # prefer smooth value if exists
    val_col = smooth_col if smooth_col in df.columns else col
    tmp = df[['date', val_col]].rename(columns={val_col:'value'})
    tmp['brand_name'] = brand
    tmp['brand_id'] = r['brand_id']
    rows.append(tmp)
if rows:
    brand_trends = pd.concat(rows, ignore_index=True, sort=False)
    brand_trends.to_csv(DATA_DIR / "brand_trends_master.csv", index=False)
    print("Saved brand_trends_master.csv", brand_trends.shape)
else:
    print("No brand trend rows found. Check trend files.")


Saved brand_trends_master.csv (16368, 4)


#### **8. Compute per-mention lift metrics from Google Trends**

In [31]:
!python3 -m pip install pandas


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


In [32]:
# 8. Compute influence metrics per mention (using Google Trends)

from scipy.stats import linregress
import numpy as np
import pandas as pd
from tqdm import tqdm

print("Loading master trend and refreshing mentions + songs...")

# reload in case notebook was restarted
brand_trends = pd.read_csv( "../data/brand_trends_master.csv", parse_dates=['date'])
mentions_df = pd.read_csv( "../data/mentions.csv")
songs_df = pd.read_csv("../data/songs_final.csv")

# --- attach release date to each mention ---
# adjust 'release_date' column name if your songs file uses something slightly different
songs_df['release_date_parsed'] = pd.to_datetime(songs_df['release_date'], errors='coerce')
mentions_final = mentions_df.merge(
    songs_df[['song_id', 'release_date_parsed']],
    on='song_id',
    how='left'
)

print("Mentions with release dates:", mentions_final['release_date_parsed'].notna().sum(), "/", len(mentions_final))

# ---- build timeseries map per brand_id ----
before_weeks = 8    # window before release
after_weeks  = 12   # window after release

brand_ts_map = {}
for bid, g in brand_trends.groupby('brand_id'):
    tmp = g[['date', 'value']].dropna().sort_values('date')
    tmp = tmp.set_index('date')
    # optional weekly resample + interpolation (smooth irregularities)
    tmp = tmp.resample('W').mean()
    tmp['value'] = tmp['value'].interpolate(limit_direction='both')
    brand_ts_map[bid] = tmp

def compute_metrics_for_row(brand_id, release_date):
    """
    For a single (brand_id, release_date) pair, compute:
        - before_avg / after_avg
        - percent_lift
        - peak_lift (max in first 3 weeks after release vs before_avg)
        - slope_change (after_slope - before_slope)
    Returns a dict or None if not enough data.
    """
    if pd.isna(release_date) or brand_id not in brand_ts_map:
        return None
    
    ts = brand_ts_map[brand_id]

    before_start = release_date - pd.Timedelta(weeks=before_weeks)
    before_end   = release_date - pd.Timedelta(days=1)
    after_start  = release_date
    after_end    = release_date + pd.Timedelta(weeks=after_weeks)

    before = ts.loc[before_start:before_end]['value']
    after  = ts.loc[after_start:after_end]['value']

    if len(before) < 2 or len(after) < 2:
        return None

    before_avg = float(before.mean())
    after_avg  = float(after.mean())
    percent_lift = float((after_avg - before_avg) / (before_avg + 1e-9) * 100)

    # peak lift in first 3 weeks after release
    peak_window = ts.loc[after_start: after_start + pd.Timedelta(weeks=3)]['value']
    peak_lift = float(peak_window.max() - before_avg) if not peak_window.empty else 0.0

    # slopes before vs after (trend change)
    bx = np.arange(len(before))
    ax = np.arange(len(after))
    b_slope = float(linregress(bx, before.values).slope) if len(bx) > 1 else 0.0
    a_slope = float(linregress(ax, after.values).slope)  if len(ax) > 1 else 0.0
    slope_change = a_slope - b_slope

    return {
        'before_avg': before_avg,
        'after_avg': after_avg,
        'percent_lift': percent_lift,
        'peak_lift': peak_lift,
        'slope_change': slope_change,
    }

# ---- loop over mentions and compute metrics ----
metric_rows = []
for _, r in tqdm(mentions_final.iterrows(), total=len(mentions_final), desc="Computing influence metrics"):
    bid = r['brand_id']
    rd  = r['release_date_parsed']
    res = compute_metrics_for_row(bid, rd)
    if res is None:
        continue
    res.update({
        'mention_id': r['mention_id'],
        'brand_id': bid,
        'brand_name': r.get('brand_name', np.nan),
        'song_id': r['song_id'],
        'artist_name': r.get('artist_name', np.nan),
        'title': r.get('title', np.nan),
        'release_date': rd
    })
    metric_rows.append(res)

metrics_df = pd.DataFrame(metric_rows)
metrics_out =  "../data/influence_metrics.csv"
metrics_df.to_csv(metrics_out, index=False)
print("Saved influence metrics:", metrics_out, "shape:", metrics_df.shape)

metrics_df.head()


Loading master trend and refreshing mentions + songs...
Mentions with release dates: 150 / 5468


Computing influence metrics: 100%|██████████| 5468/5468 [00:00<00:00, 12382.22it/s]

Saved influence metrics: ../data/influence_metrics.csv shape: (122, 12)


,before_avg,after_avg,percent_lift,peak_lift,slope_change,mention_id,brand_id,brand_name,song_id,artist_name,title,release_date
0,41.339958,40.846168,-1.194461,0.663683,-0.473992,130,26,NaN,2799,NaN,NaN,2009-01-01
1,38.807079,38.160178,-1.666966,1.112845,-0.719701,184,24,NaN,5745,NaN,NaN,2012-01-01
2,46.872612,49.811092,6.269076,0.770172,0.500929,197,61,NaN,6010,NaN,NaN,2008-01-01
3,15.046199,12.680970,-15.719775,-1.089510,-0.329654,314,28,NaN,24482,NaN,NaN,2010-01-01
4,44.912200,48.778944,8.609564,5.947940,-0.999238,315,59,NaN,24482,NaN,NaN,2010-01-01


#### **9. Gold Labels + Simple Classifier**

In [34]:
# ============================================================
# STEP 9 — HEURISTIC GOLD LABELS + SIMPLE CLASSIFIER
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import joblib

# ============================================================
# 9.1 LOAD DATA
# ============================================================

mentions = pd.read_csv("../data/mentions.csv")

# Clean missing values
mentions['sentiment'] = mentions['sentiment'].fillna("neutral").str.lower()
mentions['context'] = mentions['context'].fillna("")

# Compute context length
mentions['context_len'] = mentions['context'].apply(len)

print("Loaded mentions:", len(mentions))


# ============================================================
# 9.2 BUILD HEURISTIC GOLD LABELS
# ------------------------------------------------------------
# Labels:
#   2 = strong influence
#   1 = moderate influence
#   0 = weak / none
# ============================================================

def gold_label(row):
    # Strong: positive sentiment + long descriptive context
    if row['sentiment'] == 'positive' and row['context_len'] > 40:
        return 2
    
    # Moderate: positive OR long negative description
    if row['sentiment'] == 'positive':
        return 1
    if row['sentiment'] == 'negative' and row['context_len'] > 40:
        return 1
    
    # Otherwise weak
    return 0

mentions['gold_label'] = mentions.apply(gold_label, axis=1)

print("\nGold label distribution:")
print(mentions['gold_label'].value_counts())


# ============================================================
# 9.3 BUILD FEATURES FOR CLASSIFICATION
# ------------------------------------------------------------
# Features:
#   - context_len (numeric)
#   - sent_pos (1/0)
#   - sent_neg (1/0)
# ============================================================

mentions['sent_pos'] = (mentions['sentiment'] == 'positive').astype(int)
mentions['sent_neg'] = (mentions['sentiment'] == 'negative').astype(int)

feature_cols = ['context_len', 'sent_pos', 'sent_neg']
X = mentions[feature_cols]
y = mentions['gold_label']

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 9.4 TRAIN SIMPLE MULTI-CLASS CLASSIFIER
# ============================================================

clf = LogisticRegression(max_iter=1000, multi_class='multinomial')
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))


# ============================================================
# 9.5 SAVE MODEL + SCALER
# ============================================================

joblib.dump(clf, "../data/influence_classifier.joblib")
joblib.dump(scaler, "../data/influence_scaler.joblib")

print("\nSaved classifier + scaler to ../data/")


Loaded mentions: 5415

Gold label distribution:
gold_label
1    4441
2     847
0     127
Name: count, dtype: int64

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      0.15      0.27        26
           1       0.97      0.96      0.97       888
           2       0.81      1.00      0.90       169

    accuracy                           0.94      1083
   macro avg       0.93      0.70      0.71      1083
weighted avg       0.95      0.94      0.94      1083


Saved classifier + scaler to ../data/


/usr/local/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [35]:
mentions.to_csv("../data/mentions_with_gold_labels.csv", index=False)
print("Saved to ../data/mentions_with_gold_labels.csv")

Saved to ../data/mentions_with_gold_labels.csv


#### **10. BERT CLASSIFIER ON LYRIC CONTEXT (INFLUENCE PREDICTION)**

In [36]:
!python3.11 -m pip install --upgrade transformers
!python3.11 -m pip install accelerate
!python3.11 -m pip install transformers[torch]


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


zsh:1: no matches found: transformers[torch]


In [37]:
# ==== 10. BERT classifier to learn influence directly from lyric context ====

import pandas as pd
import numpy as np
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

# ----------------------------------------------------
# 10.1 Paths & data loading
# ----------------------------------------------------
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

mentions = pd.read_csv(DATA_DIR / "mentions.csv")
metrics_df = pd.read_csv(DATA_DIR / "influence_metrics.csv", parse_dates=["release_date"])

print("Loaded:")
print("  mentions:", mentions.shape)
print("  influence_metrics:", metrics_df.shape)

# ----------------------------------------------------
# 10.2 Rebuild gold labels (same heuristic as step 9)
# ----------------------------------------------------
def golden_label(row):
    if row["percent_lift"] > 20 or row["peak_lift"] > 15:
        return 2  # strong influence
    if row["percent_lift"] > 5 or row["slope_change"] > 0.1:
        return 1  # moderate
    return 0      # none

metrics_df["gold_label"] = metrics_df.apply(golden_label, axis=1)

# ----------------------------------------------------
# 10.3 Attach text (use mention context only)
# ----------------------------------------------------
if "context" not in mentions.columns:
    raise ValueError("mentions.csv has no 'context' column – cannot build text for BERT.")

m = metrics_df.merge(
    mentions[["mention_id", "context"]],
    on="mention_id",
    how="left"
)

# Build text column from context
m["text"] = m["context"].astype(str).str.strip()
m = m[m["text"] != ""].copy()

print("\nTotal rows with text + gold label:", len(m))
print("Gold label distribution:")
print(m["gold_label"].value_counts().sort_index())

# ----------------------------------------------------
# 10.4 Train / test split
# ----------------------------------------------------
X_text = m["text"].tolist()
y = m["gold_label"].astype(int).tolist()

# Try a stratified split; if it fails (class with <2 samples), fall back
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X_text,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
except ValueError as e:
    print("Stratified split failed:", e)
    print("Falling back to non-stratified train/test split.\n")
    X_train, X_test, y_train, y_test = train_test_split(
        X_text,
        y,
        test_size=0.2,
        random_state=42,
        stratify=None
    )

print(f"\nTrain size: {len(X_train)}  Test size: {len(X_test)}")


# ----------------------------------------------------
# 10.5 Tokenizer & encodings
# ----------------------------------------------------
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
MAX_LEN = 128

train_enc = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=MAX_LEN
)
test_enc = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=MAX_LEN
)

class InfluenceTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = InfluenceTextDataset(train_enc, y_train)
test_dataset  = InfluenceTextDataset(test_enc, y_test)

# ----------------------------------------------------
# 10.6 Model & device
# ----------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)
model.to(device)

# ----------------------------------------------------
# 10.7 Training loop (small, 1 epoch on CPU by default)
# ----------------------------------------------------
BATCH_SIZE = 16
EPOCHS = 1   # bump to 2–3 later if you want better performance

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=5e-5)

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            logits = outputs.logits
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            trues.extend(batch["labels"].cpu().numpy())
    return classification_report(trues, preds, digits=4)

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Training epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Average training loss: {avg_loss:.4f}")

    print("\nEvaluation on test set:")
    print(evaluate(model, test_loader))

# ----------------------------------------------------
# 10.8 Save model + tokenizer
# ----------------------------------------------------
save_dir = DATA_DIR / "bert_influence_model"
save_dir.mkdir(exist_ok=True, parents=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("\nSaved DistilBERT influence classifier + tokenizer to:", save_dir)


Loaded:
  mentions: (5415, 10)
  influence_metrics: (122, 12)

Total rows with text + gold label: 122
Gold label distribution:
gold_label
0    109
1     12
2      1
Name: count, dtype: int64
Stratified split failed: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.
Falling back to non-stratified train/test split.


Train size: 97  Test size: 25
Using device: cpu


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



===== Epoch 1/1 =====


Training epoch 1: 100%|██████████| 7/7 [00:59<00:00,  8.43s/it]


Average training loss: 0.6375

Evaluation on test set:


/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

           0     0.8800    1.0000    0.9362        22
           1     0.0000    0.0000    0.0000         3

    accuracy                         0.8800        25
   macro avg     0.4400    0.5000    0.4681        25
weighted avg     0.7744    0.8800    0.8238        25


Saved DistilBERT influence classifier + tokenizer to: ../data/bert_influence_model
